<a href="https://colab.research.google.com/github/borkovski-Ofer/P2-MexicoToysSales/blob/main/DataCleanning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fg-data-profiling pandas numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.3/400.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from data_profiling import ProfileReport
from google.colab import files


In [ ]:
uploaded = files.upload()

Saving data_messy.csv to data_messy.csv


In [ ]:
df= pd.read_csv('data_messy.csv')

First Step (1): load and look at your Data first

In [ ]:
profile =ProfileReport(df, title="Profiling Report", explorative=True)
profile.to_file('report.html')
files.download('report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 40.16it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

First Code: Looking at the shape|types | missing values | print head or tail of db

In [ ]:
df.shape

(105, 7)

In [ ]:
df.dtypes

,0
Ad feature,object
Type,object
Ad Appearances,int64
Cost,object
Nbr. Impressions,object
Nbr. Clicks,object
Nbr. Installs,object


In [ ]:
df.isnull().sum()

,0
Ad feature,83
Type,5
Ad Appearances,0
Cost,0
Nbr. Impressions,0
Nbr. Clicks,0
Nbr. Installs,0


In [ ]:
df.head(5)

,Ad feature,Type,Ad Appearances,Cost,Nbr. Impressions,Nbr. Clicks,Nbr. Installs
0,Background,Real life,144,"$808,218","186,109,441","615,440","145,639"
1,NaN,Blurry,96,"$237,238","56,814,047","309,305","44,832"
2,NaN,Animated,563,"$2,589,223","790,669,638","2,550,005","299,803"
3,Character Pose,Head and torso,38,"$122,087","49,616,629","226,572","33,524"
4,NaN,Entire body,43,"$76,598","53,008,121","266,473","30,470"


Next: standardised columns

In [ ]:
df.columns=(
    df.columns
    .str.replace(' ', '_')
    .str.lower()
    .str.strip()
)

In [ ]:
print(df.columns.tolist())

['ad_feature', 'type', 'ad_appearances', 'cost', 'nbr._impressions', 'nbr._clicks', 'nbr._installs']


Print precentages of missing values

In [ ]:
missing =df.isnull().mean()*100
print(missing[missing>0].sort_values(ascending=False))

ad_feature    79.047619
type           4.761905
dtype: float64


Handle the Duplicates, missing values, messy columns

In [ ]:
n_dupes = df.duplicated().sum()

In [ ]:
n_dupes

np.int64(0)

if u whant to see how this duplicated lokks like: run this

In [ ]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)

,ad_feature,type,ad_appearances,cost,nbr._impressions,nbr._clicks,nbr._installs


In [ ]:
df = df.drop_duplicates()

In [ ]:
df.shape

(105, 7)

deal with missing values:  numeric replaced with 'Median' | for category replaced blank with 'unknown'

In [ ]:
df = df.copy()
#Example:
# Numeric Columns
for col in ['total_amount']:
  df[col] = df[col].fillna(df[col].median())

  #Categorycal Columns
  categorical_cols = ['customer_id', 'country']
  df[categorical_cols]= df[categorical_cols].fillna('Unknown')

In [ ]:
 #Categorical Columns
categorical_cols = ['type']
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

Fill_Down a Col

In [ ]:
df['ad_feature'] = df['ad_feature'].ffill()

Check

In [ ]:
print(df.isnull().sum() [df.isnull().sum() > 0])

Series([], dtype: int64)


Use a mapping dictionary to standardize messy text columns -use AI to create the script

In [ ]:
 for col in ['col_1', 'col_2', 'col_3']:
  df[col] = df[col].strip().str.lower()

  #map to canonical values
  category_map = {
      'electronics': 'Electronics',
      'clothinng' : 'Clothing',
      'unknown' : 'Unknown'
  }

  col_2_map = {
      'deliverd' : 'Delivered'
  }

  df['category'] = df['category'].map(category_map).fillna(df['category'])
  df['col_2_map'] = df['col_2_map'].map(category_map).fillna(df['col_2_map'])

 using chain operation : Example: deal with '$' or ','

In [ ]:
df['unit_price'] = (
    df['unit_price']
    .astype(str)
    .str.replace(r'[$]', '', regex=True) # commas catch 1,299.00 style value
    .str.strip
)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

use a mapping dictionary to handle mixed boolean values

In [ ]:
bool_map = {
    'Yes': True, 'yes': True, 'Y': True, 'y': True,
    'TRUE': True, 'True':True, '1':True
}
df['is_returned'] = df['is_returned'].map(bool_map).fillna(False)

Fix Date Format

In [ ]:
df['order_date']= pd.to_datetime(df['order_date'] , dayfirst=False, errors='coerce')
failed = df['order_date'].isnull().sum()

In [ ]:
df.head()

Extract Day| week | month

In [ ]:
df['order_year'] = df['order_year'].dt.year
df['order_month'] = df['order_month'].dt.month
df['order_day_of_the_week'] = df['order_date'].dt.day_name()

STEP 3: Apply logical thinking to the dataset

In [ ]:
impossible_qty_mask = df['quantity'] <=0

In [ ]:
print(df.loc[impossible_qty_mask, ['col_name1', 'col_name2']])